In [27]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
import os


In [11]:
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [14]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ.get("GROK_API_KEY")
)



In [ ]:
def chat_node(state:ChatState):

    messages=state['messages']
    response=model.invoke(messages)

    return {'messages':[response]}

In [29]:
checkpointer=MemorySaver()

graph=StateGraph(ChatState)

def chat_node(state: ChatState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

graph.add_node("chat_node", chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot=graph.compile(checkpointer=checkpointer)

In [33]:


state_config = {"configurable": {"thread_id": thread_id}}

initial_state = {
    "messages": [
        HumanMessage(content="What is the capital of India?")
    ]
}

response = chatbot.invoke(initial_state, config=state_config)
print(response["messages"][-1].content)

The capital of India is **New Delhi**.


In [35]:
thread_id='1'


while True:

    user_message=input('Type here :')
    print('User :',user_message)
    if user_message.strip().lower()in ['exit','quit','bye']:
        break


    config={'configurable ': {'thread_id':thread_id}}

    config = {"configurable": {"thread_id": thread_id}}

    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]},config=config)
    print('AI:', response['messages'][-1].content)

User : hi
AI: Hello! How can I help you today?
User : my name is siddharam yalshett
AI: Nice to meet you, Siddharam Yalshett! How can I assist you today?
User : what is my name
AI: Your name is **Siddharam Yalshett**.
User : my age is 22
AI: Got it—you're 22 years old! How can I help you today?
User : what is my age
AI: You mentioned that you’re 22 years old.
User : ok exit
AI: Sure! If you need anything else later, just let me know. Have a great day!
User : exit
